# Lab 01 – Dự đoán khối lượng chim cánh cụt

**Mã nhóm** [Điền mã nhóm]  
**Thành viên** [Điền họ tên và MSSV của 4 thành viên]

Chạy các cell theo thứ tự. M1–M4 là bốn vị trí để mỗi thành viên tự chọn một thuật toán hồi quy tuyến tính. Điền các mục **TODO** khi hoàn thiện bài nhóm.

## 1. Mô tả bài toán

Mục tiêu đề xuất là dự đoán khối lượng cơ thể, `body_mass_g` (g). Bộ biến mẫu gồm chiều dài cánh bơi, chiều dài và độ sâu mỏ, loài và giới tính. Sau EDA, nhóm chốt đầu vào dùng chung cho bốn thuật toán.

**TODO** Viết động cơ chọn bài toán, câu hỏi nghiên cứu và giải thích lựa chọn bộ biến đầu vào.

## 2. Nạp và khám phá dữ liệu

Notebook đọc CSV trong thư mục `data/`. Nguồn dữ liệu là bản `penguins` đã đơn giản hóa. [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/)

In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "data" / "penguins.csv").is_file():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from source.modeling import CATEGORICAL, FEATURES, TARGET, load_penguins, regression_metrics, split_penguins
from source.models import m1, m2, m3, m4

BASE_FEATURE = "flipper_length_mm"
LABELS = {
    "flipper_length_mm": "Chiều dài cánh bơi (mm)",
    "bill_length_mm": "Chiều dài mỏ (mm)",
    "bill_depth_mm": "Độ sâu mỏ (mm)",
    "year": "Năm quan sát",
    "species": "Loài",
    "island": "Đảo",
    "sex": "Giới tính",
    "body_mass_g": "Khối lượng cơ thể (g)",
}

penguins = load_penguins()
print(f"Dữ liệu gốc: {len(penguins)} dòng, {penguins.shape[1]} cột")
display(penguins.head())
display(pd.DataFrame({
    "Kiểu dữ liệu": penguins.dtypes.astype(str),
    "Số giá trị thiếu": penguins.isna().sum(),
}))

### Chia tập và xem dữ liệu huấn luyện

Loại các dòng thiếu giá trị đích vì chúng không có nhãn để huấn luyện hoặc đánh giá. Dùng một lần chia 80/20 với seed 42 cho toàn bộ các mô hình. Pipeline học cách điền thiếu, chuẩn hóa và mã hóa trên tập huấn luyện.

**TODO** Từ các biểu đồ mẫu, khảo sát thêm số đo mỏ và phân bố khối lượng theo loài, giới tính trên `train_data`. Ghi lý do giữ hoặc loại từng cột, kiểm tra cột có sẵn lúc dự đoán, rồi cập nhật `FEATURES` trước khi sang mục 3. [Cấu hình đầu vào](modeling.py)

In [ ]:
train_data, test_data = split_penguins(penguins)
y_train = train_data[TARGET]
y_test = test_data[TARGET]

print(f"Huấn luyện: {len(train_data)} dòng | Kiểm tra: {len(test_data)} dòng")

fig, (hist_ax, scatter_ax) = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
hist_ax.hist(y_train, bins=16, color="#246b83", edgecolor="white")
hist_ax.set(xlabel=LABELS[TARGET], ylabel="Số cá thể", title="Phân bố biến đích")
scatter_ax.scatter(train_data[BASE_FEATURE], y_train, alpha=0.45, color="#246b83")
scatter_ax.set(
    xlabel=LABELS[BASE_FEATURE],
    ylabel=LABELS[TARGET],
    title="Chiều dài cánh bơi và khối lượng",
)
plt.show()

## 3. Thiết kế và huấn luyện mô hình

Điền estimator do bạn chọn vào `ESTIMATOR` trong tệp phụ trách. Notebook đánh dấu vị trí còn trống là **Chưa sẵn sàng**. OLS được huấn luyện cùng các mô hình đã khai báo để làm baseline.

Pipeline chung điền trung vị và chuẩn hóa cho biến số. Với biến phân loại, pipeline điền mức phổ biến nhất và mã hóa one-hot với một mức tham chiếu.

**TODO** Mỗi người chọn một thuật toán hồi quy tuyến tính riêng, ghi lý do chọn và các siêu tham số đã dùng. Sau khi sửa tệp, khởi động lại kernel và chạy toàn bộ notebook. [M1](models/m1.py) · [M2](models/m2.py) · [M3](models/m3.py) · [M4](models/m4.py)

In [ ]:
from source.models import ols

model_modules = (ols, m1, m2, m3, m4)
models = {}
model_info = []
for module in model_modules:
    ready = module is ols or module.ESTIMATOR is not None
    estimator = "—"
    if ready:
        model = module.build_model().fit(train_data[FEATURES], y_train)
        models[module.NAME] = model
        estimator = type(model.named_steps["regression"]).__name__
    model_info.append({
        "Mô hình": module.NAME,
        "Estimator đang chạy": estimator,
        "Trạng thái": "Sẵn sàng" if ready else "Chưa sẵn sàng",
    })
display(pd.DataFrame(model_info).set_index("Mô hình"))

## 4. Đánh giá và diễn giải

So sánh MAE, RMSE và R² của các mô hình trên cùng tập kiểm tra. MAE và RMSE tính bằng gram. R² không có đơn vị. Chọn mô hình ở danh sách bên dưới để xem hệ số và phần dư của mô hình đó.

**TODO** Viết nhận xét sau khi bốn thuật toán đã được chọn. So sánh mức chênh lệch sai số, phần dư và hệ số. Với biến số, hệ số tính theo gram trên một độ lệch chuẩn của đầu vào, giữ các biến còn lại cố định. Với biến phân loại, so sánh với mức tham chiếu hiển thị dưới bảng.

In [ ]:
predictions = {}
scores = []
for name, model in models.items():
    y_pred = model.predict(test_data[FEATURES])
    predictions[name] = y_pred
    scores.append({"Mô hình": name, **regression_metrics(y_test, y_pred)})

results = pd.DataFrame(scores).set_index("Mô hình")
display(results.round({"MAE (g)": 1, "RMSE (g)": 1, "R²": 3}))

In [ ]:
def show_diagnostics(model_name):
    model = models[model_name]
    preprocessor = model.named_steps["preprocess"]
    regression = model.named_steps["regression"]
    coefficient_model = getattr(regression, "estimator_", regression)

    if hasattr(coefficient_model, "coef_") and hasattr(coefficient_model, "intercept_"):
        intercept = np.asarray(coefficient_model.intercept_).item()
        print(f"{model_name} | Hệ số chặn: {intercept:.2f} g")
        display(pd.Series(
            np.ravel(coefficient_model.coef_),
            index=preprocessor.get_feature_names_out(),
            name="Hệ số",
        ).round(3).to_frame())
    else:
        print(f"{model_name} | Xem phần dư bên dưới và tra tài liệu estimator để trích hệ số.")

    categorical_features = [
        feature for feature in FEATURES if feature in CATEGORICAL
    ]
    if categorical_features:
        encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
        for feature, levels, reference_index in zip(
            categorical_features, encoder.categories_, encoder.drop_idx_
        ):
            print(f"Mức tham chiếu của {feature}: {levels[reference_index]}")

    y_pred = predictions[model_name]
    fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
    ax.scatter(y_pred, y_test - y_pred, alpha=0.65, color="#246b83")
    ax.axhline(0, color="#475569", linestyle="--")
    ax.set(xlabel="Dự đoán (g)", ylabel="Thực tế − dự đoán (g)", title="Phần dư trên tập kiểm tra")
    plt.show()
    plt.close(fig)

diagnostics = widgets.interactive(
    show_diagnostics, model_name=widgets.Dropdown(options=list(models), description="Mô hình"),
)
display(diagnostics)

## 5. Giao diện dự đoán

Chọn mô hình, chỉnh đầu vào rồi bấm **Dự đoán** để xem khối lượng theo gram. Khi so sánh OLS và M3, biểu đồ số đo có thêm ô chênh lệch M3 − OLS. Các đường dự đoán giữ cố định đầu vào còn lại theo biểu mẫu. Các điểm nền biểu diễn quan sát train với nhiều tổ hợp đầu vào.

[Xem hai kịch bản demo cho Section 3.4](../report/section-3-4-application.md)

In [ ]:
from source.prediction_app import PredictionApp

prediction_app = PredictionApp(models, train_data, LABELS)
display(prediction_app.panel)
prediction_app.predict_button.click()

## 6. Kết luận và tài liệu tham khảo

**TODO** Tóm tắt câu trả lời cho mục tiêu của nhóm. Dẫn số liệu từ bảng đánh giá, biểu đồ hoặc hệ số cho từng nhận xét chính. Ghi các giới hạn quan sát được và đưa phần trình bày đầy đủ vào báo cáo.

Horst, A. M., Hill, A. P., & Gorman, K. B. (2020). *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. [DOI 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218)